# 01 - Data Audit and Data Model

The Olist dataset arrives as **nine related CSV tables**, not one flat file. Before
any analysis we have to establish four things:

1. **What is the grain?** Which table is the spine, and what does one row mean?
2. **How do the tables join, and what breaks if we join them naively?**
3. **Which key identifies a customer?** The brief warns that `customer_id` is not
   a person.
4. **Where is the data missing or unusable?**

Every claim below is asserted, so re-running this notebook top to bottom is a
proof that the numbers quoted in the report still hold.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from viz import save_fig, use_report_style

use_report_style()
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from data_load import (
    CLEANING_DECISIONS, build_orders, load_all, quality_summary, zip_centroids,
)

## 1.1 The nine source tables

`load_table()` asserts the row count of every table on read, so a changed source
file fails loudly instead of silently altering the report.

In [2]:
tables = load_all()

summary = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": df.shape[1],
     "key_column": df.columns[0]}
    for name, df in tables.items()
]).sort_values("rows", ascending=False)
print(summary.to_string(index=False))
print(f"\ntotal rows across all nine tables: {summary['rows'].sum():,}")

               table    rows  columns                  key_column
         geolocation 1000163        5 geolocation_zip_code_prefix
         order_items  112650        7                    order_id
      order_payments  103886        5                    order_id
              orders   99441        8                    order_id
           customers   99441        5                 customer_id
       order_reviews   99224        7                   review_id
            products   32951        9                  product_id
             sellers    3095        4                   seller_id
category_translation      71        2       product_category_name

total rows across all nine tables: 1,550,922


## 1.2 The join trap

`order_items`, `order_payments` and `order_reviews` are all **one-to-many**
against `orders`. Merging them directly would multiply a single order into
several rows and silently reweight every average towards multi-item orders.

The cell below quantifies the damage before we avoid it.

In [3]:
orders = tables["orders"]
naive = (
    orders[["order_id"]]
    .merge(tables["order_items"], on="order_id", how="left")
    .merge(tables["order_payments"], on="order_id", how="left")
)
print(f"orders                      : {len(orders):,}")
print(f"naive orders x items x pay  : {len(naive):,}  "
      f"(+{100*(len(naive)/len(orders)-1):.0f}% rows)")

dup = naive.groupby("order_id").size()
print(f"\nworst-case row explosion for a single order: {dup.max()}x")
print(f"orders that would be counted more than once : {(dup > 1).sum():,} "
      f"({100*(dup > 1).mean():.1f}%)")
print("\nThis is why build_orders() aggregates each child table to one row per")
print("order *first*, and only then joins. The analysis grain is one order.")

orders                      : 99,441
naive orders x items x pay  : 118,434  (+19% rows)



worst-case row explosion for a single order: 63x
orders that would be counted more than once : 12,489 (12.6%)

This is why build_orders() aggregates each child table to one row per
order *first*, and only then joins. The analysis grain is one order.


## 1.3 `customer_id` is not a customer

The brief is explicit about this and it is easy to verify: `customer_id` is
regenerated for every order, so joining on it makes every order look like a
different first-time buyer.

In [4]:
cust = tables["customers"]
print(f"rows in customers table : {len(cust):,}")
print(f"distinct customer_id     : {cust['customer_id'].nunique():,}   "
      f"(one per row -> an order key, not a person)")
print(f"distinct customer_unique_id : {cust['customer_unique_id'].nunique():,}")

rep = cust.groupby("customer_unique_id").size()
print(f"\npeople who ordered more than once: {(rep > 1).sum():,} "
      f"({100*(rep > 1).mean():.1f}% of people)")
print(f"most orders by one person        : {rep.max()}")
print("\nUsing customer_id for repeat behaviour would report 0% repeat customers.")

assert cust["customer_id"].nunique() == len(cust)
assert cust["customer_unique_id"].nunique() < len(cust)

rows in customers table : 99,441
distinct customer_id     : 99,441   (one per row -> an order key, not a person)
distinct customer_unique_id : 96,096

people who ordered more than once: 2,997 (3.1% of people)
most orders by one person        : 17

Using customer_id for repeat behaviour would report 0% repeat customers.


## 1.4 Building the analysis table

`build_orders()` performs the aggregation-then-join, attaches geography, and
derives the outcome measures. The grain assertion is the important part.

In [5]:
full = build_orders(tables)
print(f"analysis table: {len(full):,} rows x {full.shape[1]} columns")
assert len(full) == len(orders), "joins must not change the grain"
print("grain preserved: one row per order\n")
print(full[["order_id", "order_status", "customer_state", "lead_category",
            "n_items", "total_price", "distance_km", "delivery_days",
            "is_late", "review_score"]].head())

analysis table: 99,441 rows x 44 columns
grain preserved: one row per order

                           order_id order_status customer_state    lead_category  n_items  total_price  distance_km  delivery_days  is_late  review_score
0  2e7a8482f6fb09756ca50c10d7bfc047      shipped             RR  furniture_decor      2.0        72.89  3198.818994            NaN      NaN           1.0
1  e5fa5a7210941f7d56d0208e4e071d35     canceled             RS        telephony      1.0        59.50   437.134129            NaN      NaN           1.0
2  809a282bbd5dbcabb6f2f724fca862ec     canceled             SP          unknown      NaN          NaN          NaN            NaN      NaN           1.0
3  bfbd0f9bdef84302105ad712db648a6c    delivered             SP    health_beauty      3.0       134.97   566.040211      54.813194      1.0           1.0
4  71303d7e93b399f5bcd537d124c0bcfa     canceled             SP             baby      1.0       100.00    10.175818            NaN      NaN           1.0

## 1.5 Missingness, and what it means

Missing values here are structural rather than random: an order that was never
delivered has no delivery date because the event never happened.

In [6]:
print("ORDER STATUS (%):")
print((full["order_status"].value_counts(normalize=True) * 100).round(2).to_string())

print("\nMISSINGNESS in the columns the analysis depends on:")
print(quality_summary(full).to_string(index=False))

undelivered = full["order_delivered_customer_date"].isna()
print(f"\norders with no delivery timestamp: {undelivered.sum():,} "
      f"({100*undelivered.mean():.2f}%)")
print("status breakdown of those orders:")
print(full.loc[undelivered, "order_status"].value_counts().to_string())

ORDER STATUS (%):
order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00

MISSINGNESS in the columns the analysis depends on:
                       column  n_missing  pct_missing  n_unique
order_delivered_customer_date       2965         2.98     95664
            order_approved_at        160         0.16     90733
                 review_score        768         0.77         5
                lead_category          0         0.00        74
                  total_price        775         0.78      7761
                total_freight        775         0.78      7970
               total_weight_g        775         0.78      2897
                  distance_km       1265         1.27     92264
            lead_payment_type          1         0.00         5
               estimated_days          0         0.00     96677

orders with no delivery timestamp: 2,965 (2

Every undelivered order is genuinely unfinished - shipped, cancelled,
unavailable or still processing. None of them is a data error, and none is
imputed: the delivery targets are simply `NaN` there.

## 1.6 Coverage: the platform ramps up

The file starts in September 2016, but the first months are a pilot. Modelling
them alongside 2018 would mix a business-scale artefact into every trend.

In [7]:
monthly = full.groupby(full["order_purchase_timestamp"].dt.to_period("M")).size()
print("orders per month, first 8 months:")
print(monthly.head(8).to_string())
print("\n... last 5 months:")
print(monthly.tail(5).to_string())

pilot = monthly[monthly.index < pd.Period("2017-01")].sum()
print(f"\ntotal orders before 2017-01-01: {pilot:,}")
print(f"typical 2018 month             : {monthly[monthly.index >= pd.Period('2018-01')].median():,.0f}")
print(f"\nanalysis window: {config.ANALYSIS_START.date()} .. {config.ANALYSIS_END.date()}")

orders per month, first 8 months:
order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
Freq: M

... last 5 months:
order_purchase_timestamp
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M

total orders before 2017-01-01: 329
typical 2018 month             : 6,620

analysis window: 2017-01-01 .. 2018-08-31


In [8]:
# ---- Figure 1: coverage and data quality ----------------------------------
fig, axes = plt.subplots(2, 1, figsize=(config.FIG_WIDTH, 5.0))

ax = axes[0]
idx = monthly.index.to_timestamp()
inside = (idx >= config.ANALYSIS_START) & (idx <= config.ANALYSIS_END)
ax.bar(idx[~inside], monthly.values[~inside], width=25,
       color=config.PALETTE["muted"], label="outside analysis window")
ax.bar(idx[inside], monthly.values[inside], width=25,
       color=config.PALETTE["primary"], label="analysis window")
ax.axvline(config.ANALYSIS_START, color="black", ls="--", lw=1)
ax.set_ylabel("orders per month")
ax.set_title("(a) Order volume: a pilot period, then a working marketplace")
ax.legend(loc="upper left")

ax = axes[1]
miss = quality_summary(full).set_index("column")["pct_missing"].sort_values()
ax.barh(miss.index, miss.values, color=config.PALETTE["accent"], alpha=0.85)
ax.set_xlabel("% of orders missing")
ax.set_title("(b) Missingness is structural, not random")
for i, v in enumerate(miss.values):
    ax.text(v + 0.05, i, f"{v:.2f}%", va="center", fontsize=7)

fig.tight_layout()
print(save_fig(fig, "fig01_coverage_quality"))

E:\2025 NUS\IT5006\IT5006 PROJECT\report\figures\fig01_coverage_quality.pdf


## 1.7 Cleaning contract

Every decision lives in `src/data_load.py::CLEANING_DECISIONS`, so the report
table and the code cannot drift apart.

In [9]:
for name, row in pd.DataFrame(CLEANING_DECISIONS).T.iterrows():
    print(f"\n### {name}")
    print(f"  issue    : {row['issue']}")
    print(f"  evidence : {row['evidence']}")
    print(f"  decision : {row['decision']}")


### one_to_many_joins
  issue    : order_items, order_payments and order_reviews are all one-to-many against orders. Joining them directly multiplies an order into several rows.
  evidence : 112,650 items and 103,886 payment records against 99,441 orders; a naive merge yields ~117k rows and silently over-weights multi-item orders in every average.
  decision : Aggregate each child table to one row per order first (counts, sums, and the attributes of the most expensive item), then join. The analysis grain is one order.

### customer_key
  issue    : customer_id is regenerated for every order, so it identifies an order-customer pair rather than a person.
  evidence : 99,441 customer_id values for 96,096 customer_unique_id values; 2,997 people ordered more than once, one of them 17 times.
  decision : Carry customer_unique_id and group on it for repeat behaviour and for train/test splitting, so the same person cannot appear on both sides of the split.

### undelivered_orders
  issue    :

In [10]:
# Final contract assertions - these are what make the notebook a proof.
assert len(tables) == 9
assert len(orders) == 99_441
assert full["order_id"].is_unique
assert full["customer_unique_id"].nunique() == 96_096
assert 2.9 < 100 * full["order_delivered_customer_date"].isna().mean() < 3.0
print("All data-contract assertions passed.")

All data-contract assertions passed.
